In [ ]:
import os

import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from lerobot.datasets import LeRobotDataset

In [ ]:
OUTPUT_DIR = "figures"

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

In [ ]:
DATASET_REPO_ID = "lerobot/pusht"
dataset = LeRobotDataset(repo_id=DATASET_REPO_ID)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=False,
    num_workers=1,
    pin_memory=(device.type == "cuda"),
    drop_last=False,
)

In [ ]:
data = next(iter(dataloader))
image = data["observation.image"][0]

## Gaussian Noise

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(10, 4))

axes[0].imshow(image.permute(1, 2, 0))
axes[0].axis("off")
axes[0].set_title("Original Image")

for idx, gaussian_noise_std in enumerate([0.1, 0.25, 0.5]):
    gaussian_noise = torch.randn_like(image) * gaussian_noise_std
    image_with_noise = image + gaussian_noise

    axes[idx + 1].imshow(image_with_noise.permute(1, 2, 0))
    axes[idx + 1].axis("off")
    axes[idx + 1].set_title(f"Gaussian Noise Std: {gaussian_noise_std}")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/gaussian_noise_examples.pdf")

## Blur

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(10, 4))

axes[0].imshow(image.permute(1, 2, 0))
axes[0].axis("off")
axes[0].set_title("Original Image")

for idx, blur_kernel_size in enumerate([3, 5, 7]):
    if blur_kernel_size == 1:
        image_with_blur = image
    else:
        image_with_blur = torch.nn.functional.avg_pool2d(
            image.unsqueeze(0), kernel_size=blur_kernel_size, stride=1, padding=blur_kernel_size // 2
        ).squeeze(0)

    axes[idx + 1].imshow(image_with_blur.permute(1, 2, 0))
    axes[idx + 1].axis("off")
    axes[idx + 1].set_title(f"Blur Kernel Size: {blur_kernel_size}")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/blur_examples.pdf")

## Brightness Shift

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(10, 4))

axes[0].imshow(image.permute(1, 2, 0))
axes[0].axis("off")
axes[0].set_title("Original Image")

for idx, brightness_shift in enumerate([-0.3, 0.05, 0.4]):
    image_with_brightness_shift = torch.clamp(image + brightness_shift, 0, 1)

    axes[idx + 1].imshow(image_with_brightness_shift.permute(1, 2, 0))
    axes[idx + 1].axis("off")
    axes[idx + 1].set_title(f"Brightness Shift: {brightness_shift}")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/brightness_shift_examples.pdf")